In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module, Linear, ReLU, Sequential, ModuleList
from torch_geometric.nn import TransformerConv, GatedGraphConv, LayerNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pickle
import os
import zipfile
import torch_geometric
import h5py
import math

# Initialize CUDA if available
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print(f"Using a single GPU or CPU.")

def load_adjacency_matrix(adj_matrix_path):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_data = pickle.load(f)

        if isinstance(adj_data, tuple):
            adj_matrix = next((np.array(item) for item in adj_data if isinstance(item, (np.ndarray, list)) and np.array(item).ndim == 2), None)
        elif isinstance(adj_data, dict):
            possible_keys = ['adj', 'adj_mx', 'matrix']
            key = next((k for k in possible_keys if k in adj_data), None)
            adj_matrix = adj_data.get(key)
        else:
            adj_matrix = np.array(adj_data)

        if adj_matrix is None or adj_matrix.ndim != 2 or adj_matrix.shape[0] != adj_matrix.shape[1]:
             raise ValueError("Could not extract a valid square 2D matrix.")

        adj_matrix = adj_matrix.astype(np.float32)
        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}")
        return adj_matrix

    except Exception as e:
        print(f"Error loading adjacency matrix from {adj_matrix_path}: {str(e)}")
        raise

def load_pems_bay_data(data_dir='PEMS-BAY', num_days=99, custom_adj_path=None):
    if custom_adj_path:
        print(f"Loading custom adjacency matrix from: {custom_adj_path}")
        adj_mat = load_adjacency_matrix(custom_adj_path)
    else:
        raise ValueError("A custom adjacency matrix path must be provided for PEMS-BAY.")

    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        node_values = np.array(f['speed']['block0_values'])

    time_steps = num_days * 288
    node_values = node_values[:time_steps]
    node_values = node_values.T

    print(f"Loaded node data for {num_days} days, shape: {node_values.shape}")

    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T

    edge_index = torch.tensor(np.where(adj_mat > 0), dtype=torch.long)
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)

    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)

    return node_values, edge_index, edge_weight, scaler

def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = np.array(features, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)
    features = torch.tensor(features).permute(0, 2, 1)
    labels = torch.tensor(labels).permute(0, 2, 1)
    return features, labels

def create_100th_day_data(data_dir='PEMS-BAY', in_timestep=12, out_timestep=12):
    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        node_values = np.array(f['speed']['block0_values'])
    
    start_idx = 28512
    end_idx = start_idx + 288
    
    if node_values.shape[0] < end_idx:
        print(f"Warning: Dataset has {node_values.shape[0]} timesteps, insufficient for 100th day. Using last 288 timesteps.")
        start_idx = max(0, node_values.shape[0] - 288)
        node_values = node_values[start_idx:start_idx + 288]
    else:
        node_values = node_values[start_idx:end_idx]

    node_values = node_values.T

    scaler_100 = MinMaxScaler()
    node_values = scaler_100.fit_transform(node_values.T).T
    
    features, labels = create_data_samples(node_values, in_timestep, out_timestep)
    return features, labels, scaler_100

def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction.png'):
    os.makedirs(output_dir, exist_ok=True)
    pred = pred[0].cpu().numpy().T
    target = target[0].cpu().numpy().T
    pred = scaler.inverse_transform(pred).T
    target = scaler.inverse_transform(target).T
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

class GPSLayer(nn.Module):
    def __init__(self, hidden_channels, heads=4, dropout=0.1):
        super().__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout)
        self.norm1 = LayerNorm(hidden_channels)
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_trans = self.transformer(x, edge_index)
        x = x + self.dropout(x_gcn) + self.dropout(x_trans)
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x

class GPSGraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([GPSLayer(hidden_channels, heads=heads) for _ in range(num_layers)])
        self.norm = LayerNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        x = self.node_emb(x)
        if self.use_pe and pe is not None:
            x = torch.cat([x, self.pe_lin(pe)], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

# Main workflow
adj_matrices = [
    'adj_mx_knn_lane_pems.pkl',
    'adj_mx_knn_pems.pkl',
    'adj_mx_radius_lane_pems.pkl',
    'adj_mx_radius_pems.pkl'
]
results = {}

for adj_file in adj_matrices:
    print(f"\n--- Processing Adjacency Matrix: {adj_file} ---")
    
    try:
        node_values, edge_index, edge_weight, scaler = load_pems_bay_data(custom_adj_path=f'PEMS-BAY/{adj_file}')
        in_timestep, out_timestep = 12, 12

        features, labels = create_data_samples(node_values, in_timestep, out_timestep)

        total_samples = len(features)
        train_end = int(0.6 * total_samples)
        val_end = train_end + int(0.2 * total_samples)

        train_features, train_labels = features[:train_end], labels[:train_end]
        val_features, val_labels = features[train_end:val_end], labels[train_end:val_end]
        test_features, test_labels = features[val_end:], labels[val_end:]
        print(f"Total Samples: {total_samples} -> Train: {len(train_features)}, Val: {len(val_features)}, Test: {len(test_features)}")

        transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
        def create_pyg_dataset(features, labels):
            return [transform(Data(x=f.T, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(features, labels)]

        train_data = create_pyg_dataset(train_features, train_labels)
        val_data = create_pyg_dataset(val_features, val_labels)
        test_data = create_pyg_dataset(test_features, test_labels)
        
        day_100_features, day_100_labels, day_100_scaler = create_100th_day_data()
        day_100_data = create_pyg_dataset(day_100_features, day_100_labels)

        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)

        model = GPSGraphTransformer(in_channels=in_timestep, hidden_channels=64, out_channels=out_timestep).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        criterion = nn.MSELoss()

        print("\n--- Starting Training ---")
        for epoch in range(20):
            model.train()
            total_train_loss = 0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                loss = criterion(out, target)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            with torch.no_grad():
                total_val_loss = 0
                for d in val_loader:
                    d = d.to(device)
                    out = model(d.x, d.edge_index, d.edge_attr, d.pe, d.batch)
                    target = d.y.view(d.batch.max().item() + 1, out.shape[1], -1)
                    total_val_loss += criterion(out, target).item()
            
            print(f"Epoch {epoch + 1:02d}, Train Loss: {total_train_loss / len(train_loader):.4f}, Val Loss: {total_val_loss / len(val_loader):.4f}")

        print("\n--- Evaluating Model ---")
        def evaluate(loader, scaler_obj, set_name):
            model.eval()
            total_mae, total_rmse, total_mape, count = 0, 0, 0, 0
            with torch.no_grad():
                for i, data in enumerate(loader):
                    data = data.to(device)
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    mae, rmse, mape = compute_metrics(out, target, scaler_obj)
                    total_mae += mae
                    total_rmse += rmse
                    total_mape += mape
                    count += 1
                    if i == 0: visualize_predictions(out, target, scaler_obj, f'plots_{adj_file.split(".")[0]}', filename=f'preds_{set_name}_{adj_file}.png')
            
            print(f"{set_name} Results -> MAE: {total_mae/count:.4f}, RMSE: {total_rmse/count:.4f}, MAPE: {total_mape/count:.4f}%")
            return {'mae': total_mae/count, 'rmse': total_rmse/count, 'mape': total_mape/count}

        test_metrics = evaluate(test_loader, scaler, "Test")
        day_100_metrics = evaluate(day_100_loader, day_100_scaler, "100th Day")
        
        results[adj_file] = {'test': test_metrics, '100th_day': day_100_metrics}

    except Exception as e:
        print(f"An error occurred for {adj_file}: {e}")
        results[adj_file] = {'test': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')},
                                                       '100th_day': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')}}

# --- Final Summary ---
print("\n--- Summary of All Results ---")
for key, metrics_dict in results.items():
    print(f"\nAdjacency Matrix: {key}")
    print(f"  Test      - MAE: {metrics_dict['test']['mae']:.4f}, RMSE: {metrics_dict['test']['rmse']:.4f}, MAPE: {metrics_dict['test']['mape']:.4f}%")
    print(f"  100th Day - MAE: {metrics_dict['100th_day']['mae']:.4f}, RMSE: {metrics_dict['100th_day']['rmse']:.4f}, MAPE: {metrics_dict['100th_day']['mape']:.4f}%")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!

--- Processing Adjacency Matrix: adj_mx_knn_lane_pems.pkl ---
Loading custom adjacency matrix from: PEMS-BAY/adj_mx_knn_lane_pems.pkl
Loaded adjacency matrix from PEMS-BAY/adj_mx_knn_lane_pems.pkl, shape: (325, 325)
Loaded node data for 99 days, shape: (325, 28512)


/tmp/ipykernel_92146/3044618624.py:82: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  edge_index = torch.tensor(np.where(adj_mat > 0), dtype=torch.long)


Total Samples: 28489 -> Train: 17093, Val: 5697, Test: 5699

--- Starting Training ---
Epoch 01, Train Loss: 0.0144, Val Loss: 0.0085
Epoch 02, Train Loss: 0.0057, Val Loss: 0.0075
Epoch 03, Train Loss: 0.0054, Val Loss: 0.0075
Epoch 04, Train Loss: 0.0051, Val Loss: 0.0069
Epoch 05, Train Loss: 0.0050, Val Loss: 0.0068
Epoch 06, Train Loss: 0.0049, Val Loss: 0.0061
Epoch 07, Train Loss: 0.0048, Val Loss: 0.0062
Epoch 08, Train Loss: 0.0048, Val Loss: 0.0061
Epoch 09, Train Loss: 0.0048, Val Loss: 0.0058
Epoch 10, Train Loss: 0.0047, Val Loss: 0.0058
Epoch 11, Train Loss: 0.0046, Val Loss: 0.0062
Epoch 12, Train Loss: 0.0046, Val Loss: 0.0060
Epoch 13, Train Loss: 0.0045, Val Loss: 0.0057
Epoch 14, Train Loss: 0.0044, Val Loss: 0.0058
Epoch 15, Train Loss: 0.0044, Val Loss: 0.0054
Epoch 16, Train Loss: 0.0043, Val Loss: 0.0058
Epoch 17, Train Loss: 0.0042, Val Loss: 0.0057
Epoch 18, Train Loss: 0.0041, Val Loss: 0.0055
Epoch 19, Train Loss: 0.0041, Val Loss: 0.0054
Epoch 20, Train Loss